In [ ]:
import torch
x = torch.tensor([1.0, 2.0, 3.0])
x = x.to("cuda")
w = torch.tensor(2.0, requires_grad=True)
y = w * w
y.backward()
print(w.grad)

tensor(1.6000)


In [4]:
w = torch.tensor(2.0, requires_grad=True)
lr = 0.1

for step in range(5):
    loss = w * w
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
    w.grad.zero_()
    print(step, round(w.item(), 4), round(loss.item(), 4))

0 1.6 4.0
1 1.28 2.56
2 1.024 1.6384
3 0.8192 1.0486
4 0.6554 0.6711


In [7]:
import torch.nn as nn

layer = nn.Linear(3, 1)
print(layer.weight.shape, layer.bias.shape)
print(sum(p.numel() for p in layer.parameters()))

torch.Size([1, 3]) torch.Size([1])
4


In [10]:
import torch
import torch.nn as nn

class SmallNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(3, 4)
        self.out = nn.Linear(4, 1)

    def forward(self, x):
        h = torch.relu(self.hidden(x))
        return self.out(h)

model = SmallNet()
x = torch.tensor([[1.0, 2.0, 3.0]])
print(model(x))
print(sum(p.numel() for p in model.parameters()))
x2 = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(model(x2).shape)

tensor([[0.2766]], grad_fn=<AddmmBackward0>)
21
torch.Size([2, 1])


In [27]:
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
X = torch.randn(200, 3)
y = (2 * X[:, 0] - X[:, 1] + 0.5 * X[:, 2]).unsqueeze(1)

loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

model = SmallNet()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(20):
    total = 0.0
    for xb, yb in loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
    if  epoch %5 == 0  or epoch == 19:
        print(epoch, round(total / len(X), 4))

0 4.6509
5 1.8662
10 0.203
15 0.097
19 0.077


In [ ]:
from torch.utils.data import Dataset

class PCBDataset(Dataset):
    def __init__(self, csv_path, raw_dir):
        self.df = pd.read_csv(csv_path)
        self.raw_dir = raw_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = cv2.imread(str(self.raw_dir / row["test_img"]), cv2.IMREAD_GRAYSCALE)
        boxes = np.loadtxt(self.raw_dir / row["ann"], dtype=int, ndmin=2)
        return torch.from_numpy(img), torch.from_numpy(boxes)